# GCF Project Evaluation Pipeline
**Evaluation Quant Specialist | Green Climate Fund**

---
**Workflow:**
1. `Section 1` — Fetch & cache all projects (single API call; ~2.5MB; reloads from cache)
2. `Section 2` — Data inventory + unnest nested fields (Countries, Entities)
3. `Section 3` — Portfolio tabulation (region, sector, status, size, risk, entity)
4. `Section 4` — Evaluation question templates (duplicate per question)
5. `Section 5` — FT-style visualizations
6. `Section 6` — Narrative generation + Excel export

**API fields confirmed from live data:**
`ProjectsID, ApprovedRef, ProjectName, ApprovalDate, Theme, Sector, Status, Size,`  
`RiskCategory, TotalGCFFunding, TotalCoFinancing, TotalValue, LifeTimeCO2,`  
`DirectBeneficiaries, IndirectBeneficiaries, DurationMonths, BoardMeeting`  
`+ nested: Countries[CountryName, Region, LDCs, SIDS], Entities[Name, Acronym, Access, Type]`


## Setup

In [1]:
import os, json, warnings
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

API_URL    = 'http://api.gcfund.org/v1/projects'
CACHE_DIR  = Path('gcf_cache')
OUT_DIR    = Path('gcf_output')
CACHE_FILE = CACHE_DIR / 'projects_raw.parquet'
FORCE_REFRESH = False   # set True to re-fetch from API

CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
print('Config OK')

ModuleNotFoundError: No module named 'requests'

## Section 1 — Fetch & Cache
_The API returns all projects in a single response (~354 records). Cached to parquet after first run._

In [ ]:
def fetch_projects(url: str) -> list[dict]:
    """Fetch all GCF projects. API returns a flat JSON array — no pagination needed."""
    resp = requests.get(url, headers={'Accept': 'application/json'}, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    # API returns either a direct list or an envelope
    if isinstance(data, list):
        return data
    return (data.get('projects') or data.get('data') or
            data.get('items') or data.get('results') or [])


if not FORCE_REFRESH and CACHE_FILE.exists():
    df_raw = pd.read_parquet(CACHE_FILE)
    print(f'Loaded from cache: {len(df_raw):,} projects | {df_raw.shape[1]} fields')
    mtime = datetime.fromtimestamp(CACHE_FILE.stat().st_mtime)
    print(f'Cache date: {mtime:%Y-%m-%d %H:%M}')
else:
    print('Fetching from API (allow ~30s)…')
    records = fetch_projects(API_URL)
    df_raw = pd.json_normalize(records, sep='_')
    df_raw.columns = [c.lower() for c in df_raw.columns]
    df_raw.to_parquet(CACHE_FILE, index=False)
    print(f'Fetched & cached: {len(df_raw):,} projects | {df_raw.shape[1]} fields')

df = df_raw.copy()
print(f'\nColumns: {list(df.columns)}')

## Section 2 — Data Inventory & Field Preparation
_Unnests nested fields (Countries, Entities) and converts types. Run this before any analysis._

In [ ]:
# ── Type conversions ───────────────────────────────────────────────────────
# Dates
for col in ['approvaldate', 'startdate', 'enddate', 'dateimplementationstart',
            'dateclosing', 'datecompletion', 'datecancelled']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

df['approval_year'] = df['approvaldate'].dt.year

# Numerics
for col in ['totalgcffunding', 'totalcofinancing', 'totalvalue',
            'lifetimeco2', 'directbeneficiaries', 'indirectbeneficiaries', 'durationmonths']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Co-financing leverage ratio
df['cofinancing_leverage'] = df['totalcofinancing'] / df['totalgcffunding']

# Total beneficiaries
df['total_beneficiaries'] = df['directbeneficiaries'].fillna(0) + df['indirectbeneficiaries'].fillna(0)

print('Type conversions done.')

In [ ]:
# ── Unnest Countries — direct extraction (robust to column-naming variations) ──
def extract_first_country(val):
    if isinstance(val, list) and val:
        return val[0]
    if isinstance(val, str):
        try:
            parsed = json.loads(val)
            if isinstance(parsed, list) and parsed:
                return parsed[0]
        except Exception:
            pass
    return {}

country_first = df['countries'].apply(extract_first_country)
df['country_name'] = country_first.apply(lambda x: x.get('CountryName') if isinstance(x, dict) else None)
df['region']       = country_first.apply(lambda x: x.get('Region')      if isinstance(x, dict) else None)
df['iso3']         = country_first.apply(lambda x: x.get('ISO3')        if isinstance(x, dict) else None)
df['is_ldc']       = country_first.apply(lambda x: x.get('LDCs')        if isinstance(x, dict) else None)
df['is_sids']      = country_first.apply(lambda x: x.get('SIDS')        if isinstance(x, dict) else None)

def count_countries(val):
    if isinstance(val, list): return len(val)
    if isinstance(val, str):
        try: return len(json.loads(val))
        except: pass
    return 1

df['n_countries']    = df['countries'].apply(count_countries)
df['is_multicountry'] = df['n_countries'] > 1

print(f'Countries extracted. {df["country_name"].notna().sum()} projects with country.')
print(f'Regions: {df["region"].value_counts().to_dict()}')


In [ ]:
# ── Unnest Entities — direct extraction ──────────────────────────────────
def extract_first_entity(val):
    if isinstance(val, list) and val:
        return val[0]
    if isinstance(val, str):
        try:
            parsed = json.loads(val)
            if isinstance(parsed, list) and parsed:
                return parsed[0]
        except: pass
    return {}

entity_first = df['entities'].apply(extract_first_entity)
df['entity_name']    = entity_first.apply(lambda x: x.get('Name')    if isinstance(x, dict) else None)
df['entity_acronym'] = entity_first.apply(lambda x: x.get('Acronym') if isinstance(x, dict) else None)
df['entity_access']  = entity_first.apply(lambda x: x.get('Access')  if isinstance(x, dict) else None)
df['entity_type']    = entity_first.apply(lambda x: x.get('Type')    if isinstance(x, dict) else None)

print(f'Entities extracted. Access types: {df["entity_access"].value_counts().to_dict()}')


In [ ]:
# ── Field inventory ────────────────────────────────────────────────────────
# Exclude raw nested columns for cleaner view
analysis_cols = [c for c in df.columns if c not in ['countries', 'entities', 'disbursements', 'funding', 'resultareas']]
inv_df = df[analysis_cols]

inventory = pd.DataFrame({
    'field': inv_df.columns,
    'dtype': inv_df.dtypes.values,
    'completeness_%': (inv_df.notna().mean() * 100).round(1).values,
    'n_unique': inv_df.nunique().values,
    'sample': [str(inv_df[c].dropna().iloc[0])[:55] if inv_df[c].notna().any() else 'NULL' for c in inv_df.columns]
}).sort_values('completeness_%', ascending=False).reset_index(drop=True)

display(HTML('<h4>Field Inventory (analysis-ready columns)</h4>'))
display(inventory)

## Section 3 — Portfolio Tabulation
_Standard cross-tabs. Add new ones by copying a `portfolio_table()` call._

In [ ]:
def portfolio_table(data, group_by, amount_col='totalgcffunding', label=None, top_n=None):
    """Cross-tab: count + GCF funding + co-financing + share by any dimension."""
    if group_by not in data.columns:
        print(f'Column {group_by!r} not in dataframe'); return
    agg = {
        'Projects': ('projectsid', 'count'),
        'GCF_M':    (amount_col, lambda x: round(x.sum() / 1e6, 1)),
        'Avg_M':    (amount_col, lambda x: round(x.mean() / 1e6, 1)),
        'Share_%':  (amount_col, lambda x: round(x.sum() / data[amount_col].sum() * 100, 1)),
    }
    if 'totalcofinancing' in data.columns:
        agg['CoFin_M']   = ('totalcofinancing', lambda x: round(x.sum() / 1e6, 1))
        agg['Leverage']  = ('cofinancing_leverage', lambda x: round(x.median(), 2))
    tbl = data.groupby(group_by).agg(**agg)
    tbl = tbl.sort_values('GCF_M', ascending=False)
    if top_n: tbl = tbl.head(top_n)
    display(HTML(f'<h4>Portfolio by {label or group_by.replace("_"," ").title()}</h4>'))
    display(tbl)
    return tbl

# ── Standard tabulations ────────────────────────────
portfolio_table(df, 'region',       label='Region')
portfolio_table(df, 'theme',        label='Theme (Mitigation / Adaptation / Cross-cutting)')
portfolio_table(df, 'sector',       label='Sector (Public / Private)')
portfolio_table(df, 'status',       label='Project Status')
portfolio_table(df, 'size',         label='Project Size')
portfolio_table(df, 'riskcategory', label='Risk Category')

In [ ]:
# ── Top 15 Accredited Entities ─────────────────────────────────────────────
if 'entity_name' in df.columns:
    portfolio_table(df, 'entity_name', label='Accredited Entity', top_n=15)

# ── Direct vs International Access ────────────────────────────────────────
if 'entity_access' in df.columns:
    portfolio_table(df, 'entity_access', label='Access Modality')

# ── LDC / SIDS splits ─────────────────────────────────────────────────────
if 'is_ldc' in df.columns:
    portfolio_table(df, 'is_ldc', label='Least Developed Countries (LDC)')
if 'is_sids' in df.columns:
    portfolio_table(df, 'is_sids', label='Small Island Developing States (SIDS)')

In [ ]:
# ── Yearly approval trend ─────────────────────────────────────────────────
trend = df.groupby('approval_year').agg(
    Projects=('projectsid', 'count'),
    GCF_M=('totalgcffunding', lambda x: round(x.sum() / 1e6, 1)),
    CoFin_M=('totalcofinancing', lambda x: round(x.sum() / 1e6, 1)),
    Avg_M=('totalgcffunding', lambda x: round(x.mean() / 1e6, 1)),
    Beneficiaries=('total_beneficiaries', 'sum'),
    CO2_Mt=('lifetimeco2', lambda x: round(x.sum() / 1e6, 2))
).dropna(subset=['Projects'])

display(HTML('<h4>Annual Approval Trend</h4>'))
display(trend)

## Section 4 — Portfolio Risk Analysis
_IMF / quant-economist framework: concentration, instrument structure, at-risk, vintage cohort, leverage volatility. Duplicate cells to add evaluation questions._

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK SETUP — instrument-level funding table (run before any Risk EQ)
# ═══════════════════════════════════════════════════════════════════════
def gini(arr):
    """Gini coefficient (0=equal, 1=total concentration)."""
    a = np.sort(arr[~np.isnan(arr)])
    if len(a) == 0 or a.sum() == 0: return np.nan
    n = len(a)
    return (2 * np.sum(np.arange(1, n+1) * a) - (n+1) * a.sum()) / (n * a.sum())

def hhi(arr):
    """Herfindahl-Hirschman Index (0-10000). >2500 = high concentration."""
    a = np.array(arr, dtype=float)
    if a.sum() == 0: return np.nan
    s = a / a.sum() * 100
    return round(float((s**2).sum()), 1)

def effective_n(arr):
    """Effective number of entities = 1 / sum(shares^2)."""
    a = np.array(arr, dtype=float)
    if a.sum() == 0: return np.nan
    s = a / a.sum()
    denom = float((s**2).sum())
    return round(1 / denom, 1) if denom > 0 else np.nan

def parse_list_col(val):
    """Safely coerce a column value to a Python list."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try: return json.loads(val)
        except: return []
    try: return list(val)
    except: return []

# Explode Funding nested list into instrument-level rows
fund_rows = []
for _, row in df.iterrows():
    items = parse_list_col(row.get('funding', []))
    for f in items:
        if not isinstance(f, dict): continue
        fund_rows.append({
            'approvedref':     row.get('approvedref'),
            'sector':          row.get('sector'),
            'theme':           row.get('theme'),
            'region':          row.get('region'),
            'country_name':    row.get('country_name'),
            'entity_name':     row.get('entity_name'),
            'totalgcffunding': row.get('totalgcffunding'),
            'approval_year':   row.get('approval_year'),
            'status':          row.get('status'),
            'source':          f.get('Source'),
            'instrument':      f.get('Instrument'),
            'budget_usd':      pd.to_numeric(f.get('BudgetUSDeq') or f.get('Budget'), errors='coerce'),
            'currency':        f.get('Currency'),
        })

if not fund_rows:
    print('WARNING: No funding rows found. Check the funding column.')
    fund_df = pd.DataFrame(columns=['approvedref','source','instrument','budget_usd',
                                    'sector','theme','region','country_name','approval_year'])
else:
    fund_df = pd.DataFrame(fund_rows)

gcf_fund = fund_df[fund_df['source'] == 'GCF'].copy() if 'source' in fund_df.columns else fund_df.iloc[0:0]
cofin_df = fund_df[fund_df['source'] == 'Co-Financing'].copy() if 'source' in fund_df.columns else fund_df.iloc[0:0]

print(f'Instrument table: {len(fund_df):,} rows | GCF: {len(gcf_fund)} | Co-Fin: {len(cofin_df)}')

if len(gcf_fund):
    display(HTML('<h4>GCF Instrument Mix</h4>'))
    inst_summary = gcf_fund.groupby('instrument').agg(
        Projects=('approvedref', 'nunique'),
        GCF_M=('budget_usd', lambda x: round(x.sum()/1e6, 1)),
        Share_pct=('budget_usd', lambda x: round(x.sum()/gcf_fund['budget_usd'].sum()*100, 1)),
        Avg_M=('budget_usd', lambda x: round(x.mean()/1e6, 1)),
    ).sort_values('GCF_M', ascending=False)
    display(inst_summary)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ1: Portfolio Concentration Risk
# HHI, Gini, effective-N, tail exposure by country / region / entity
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ1: Portfolio Concentration Risk</h3>'))

amt = df['totalgcffunding'].dropna()

hhi_country = hhi(df.groupby('country_name')['totalgcffunding'].sum().values)
hhi_region  = hhi(df.groupby('region')['totalgcffunding'].sum().values)
hhi_entity  = hhi(df.groupby('entity_name')['totalgcffunding'].sum().values)
eff_country = effective_n(df.groupby('country_name')['totalgcffunding'].sum().values)
eff_entity  = effective_n(df.groupby('entity_name')['totalgcffunding'].sum().values)
gini_coef   = round(gini(amt.values), 4)

top1  = amt.nlargest(1).sum()  / amt.sum()
top5  = amt.nlargest(5).sum()  / amt.sum()
top10 = amt.nlargest(10).sum() / amt.sum()

risk_eq1 = {
    'HHI — by country  (>2500 = high concentration)': hhi_country,
    'HHI — by region':                                 hhi_region,
    'HHI — by entity':                                 hhi_entity,
    'Effective N — countries':                         eff_country,
    'Effective N — entities':                          eff_entity,
    'Gini coefficient (0=equal, 1=concentrated)':      gini_coef,
    'Top-1 project share %':                           f'{top1:.1%}',
    'Top-5 projects share %':                          f'{top5:.1%}',
    'Top-10 projects share %':                         f'{top10:.1%}',
    '99th pct project size (USD M)':                   f"${amt.quantile(0.99)/1e6:,.1f}M",
    '95th pct project size (USD M)':                   f"${amt.quantile(0.95)/1e6:,.1f}M",
}
display(pd.DataFrame.from_dict(risk_eq1, orient='index', columns=['Value']))

portfolio_table(df, 'country_name', label='Concentration — Top 20 Countries', top_n=20)
portfolio_table(df, 'entity_name',  label='Concentration — Top 15 Entities',  top_n=15)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ2: Instrument Structure & Repayment Risk
# Grant vs loan vs equity vs guarantee mix; where is debt exposure?
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ2: Instrument Structure & Repayment Risk</h3>'))

# Instrument x Sector matrix
gcf_by_inst = gcf_fund.groupby(['instrument', 'sector']).agg(
    Projects=('approvedref', 'nunique'),
    GCF_M=('budget_usd', lambda x: round(x.sum()/1e6, 1)),
).unstack(fill_value=0)
gcf_by_inst.columns = [f'{col[1]}_{col[0]}' for col in gcf_by_inst.columns]
display(HTML('<h4>GCF Instrument x Sector (USD M)</h4>'))
display(gcf_by_inst)

# Instrument risk buckets
debt_instruments  = ['Senior Loans', 'Subordinated Loans']
grant_instruments = ['Grants', 'Reimbursable Grants']
debt_df    = gcf_fund[gcf_fund['instrument'].isin(debt_instruments)]
grant_df   = gcf_fund[gcf_fund['instrument'].isin(grant_instruments)]
equity_df  = gcf_fund[gcf_fund['instrument'] == 'Equity']
guar_df    = gcf_fund[gcf_fund['instrument'] == 'Guarantees']
total_gcf  = gcf_fund['budget_usd'].sum()

summary = {
    'Total GCF committed (USD M)':               round(total_gcf/1e6, 1),
    'Grants + Reimbursable Grants (USD M)':       round(grant_df['budget_usd'].sum()/1e6, 1),
    '  — grant share':                            f"{grant_df['budget_usd'].sum()/total_gcf:.1%}",
    'Senior + Subordinated Loans (USD M)':        round(debt_df['budget_usd'].sum()/1e6, 1),
    '  — debt share (repayment exposure)':        f"{debt_df['budget_usd'].sum()/total_gcf:.1%}",
    'Equity (USD M)':                             round(equity_df['budget_usd'].sum()/1e6, 1),
    'Guarantees (USD M)':                         round(guar_df['budget_usd'].sum()/1e6, 1),
    'Projects with any loan instrument':          len(debt_df['approvedref'].unique()),
    'Projects with equity instrument':            len(equity_df['approvedref'].unique()),
}
display(HTML('<h4>Instrument Risk Summary</h4>'))
display(pd.DataFrame.from_dict(summary, orient='index', columns=['Value']))

# Loan exposure by region
if len(debt_df):
    display(HTML('<h4>Loan Exposure by Region</h4>'))
    display(debt_df.groupby('region').agg(
        Loan_projects=('approvedref','nunique'),
        Loan_M=('budget_usd', lambda x: round(x.sum()/1e6, 1))
    ).sort_values('Loan_M', ascending=False))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ3: Implementation (At-Risk) Portfolio
# Cancellation rate, overdue projects, portfolio-at-risk
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ3: Implementation Risk — Portfolio-at-Risk</h3>'))

risk_map = {
    'Under Implementation': 'Active',
    'Operationally Closed':  'Closed',
    'Financially Closed':    'Closed',
    'Approved':              'Pipeline',
    'Cancelled':             'At-Risk',
}
df['status_class'] = df['status'].map(risk_map).fillna('Other')

display(HTML('<h4>Status Classification</h4>'))
status_tbl = df.groupby('status_class').agg(
    Projects=('projectsid', 'count'),
    GCF_M=('totalgcffunding', lambda x: round(x.sum()/1e6, 1)),
    Share_pct=('totalgcffunding', lambda x: round(x.sum()/df['totalgcffunding'].sum()*100, 1))
).sort_values('GCF_M', ascending=False)
display(status_tbl)

# Cancellation indicators
cancelled = df[df['status'] == 'Cancelled']
cancel_indicators = {
    'Total projects':                          len(df),
    'Cancelled projects':                      len(cancelled),
    'Cancellation rate':                       f'{len(cancelled)/len(df):.1%}',
    'Cancelled GCF (USD M)':                   round(cancelled['totalgcffunding'].sum()/1e6, 1),
    'Cancelled share of total GCF':            f"{cancelled['totalgcffunding'].sum()/df['totalgcffunding'].sum():.1%}",
}
display(HTML('<h4>Cancellation Risk</h4>'))
display(pd.DataFrame.from_dict(cancel_indicators, orient='index', columns=['Value']))

# Overdue active projects
impl = df[df['status'] == 'Under Implementation'].copy()
impl['days_since_approval'] = (pd.Timestamp.now() - impl['approvaldate'].dt.tz_localize(None)).dt.days
impl['planned_duration_days'] = impl['durationmonths'] * 30.4
impl['time_overrun_days'] = impl['days_since_approval'] - impl['planned_duration_days']
impl['is_overdue'] = impl['time_overrun_days'] > 0

overdue_indicators = {
    'Active (under implementation) projects':         len(impl),
    'Overdue (beyond planned duration)':              int(impl['is_overdue'].sum()),
    'Overdue rate':                                   f"{impl['is_overdue'].mean():.1%}",
    'Avg overrun — overdue projects (days)':          round(impl[impl['is_overdue']]['time_overrun_days'].mean(), 0),
    'GCF in overdue projects (USD M)':                round(impl[impl['is_overdue']]['totalgcffunding'].sum()/1e6, 1),
}
display(HTML('<h4>Implementation Duration Risk</h4>'))
display(pd.DataFrame.from_dict(overdue_indicators, orient='index', columns=['Value']))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ4: Vintage Cohort Analysis
# Compare approval-year cohorts on scale, leverage, risk, and instrument mix
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ4: Vintage Cohort Analysis</h3>'))

cohort = df.groupby('approval_year').agg(
    N_projects=('projectsid', 'count'),
    GCF_M=('totalgcffunding', lambda x: round(x.sum()/1e6, 1)),
    Avg_M=('totalgcffunding', lambda x: round(x.mean()/1e6, 1)),
    Median_leverage=('cofinancing_leverage', 'median'),
    Pct_cancelled=('status', lambda x: round((x=='Cancelled').mean()*100, 1)),
    Pct_active=('status', lambda x: round((x=='Under Implementation').mean()*100, 1)),
    Pct_private=('sector', lambda x: round((x.str.lower()=='private').mean()*100, 1)),
    HiRisk_pct=('riskcategory', lambda x: round((x.isin(['Category A','Category B'])).mean()*100, 1)),
).round(2)
display(cohort)

# Instrument mix by cohort
cohort_inst = (gcf_fund.groupby(['approval_year', 'instrument'])['budget_usd']
               .sum().unstack(fill_value=0) / 1e6).round(1)
cohort_inst.columns.name = None
display(HTML('<h4>GCF Instrument Mix by Approval Year (USD M)</h4>'))
display(cohort_inst)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ5: Co-financing Leverage — Distribution & Crowding-in Risk
# Is leverage driven by outliers? Which instruments crowd-in most?
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ5: Co-financing Leverage Risk</h3>'))

lev = df['cofinancing_leverage'].dropna()

leverage_dist = {
    'N projects with co-financing':                    int(lev.notna().sum()),
    'Median leverage (x)':                             round(lev.median(), 2),
    'Mean leverage (x)':                               round(lev.mean(), 2),
    '25th pct leverage (x)':                           round(lev.quantile(0.25), 2),
    '75th pct leverage (x)':                           round(lev.quantile(0.75), 2),
    '95th pct leverage (x)':                           round(lev.quantile(0.95), 2),
    'Std dev':                                         round(lev.std(), 2),
    'CoV (std/mean) — leverage volatility':            round(lev.std()/lev.mean(), 3),
    'Projects with leverage < 0.5x':                   int((lev < 0.5).sum()),
    'Projects with leverage > 5x (outlier)':           int((lev > 5).sum()),
}
display(pd.DataFrame.from_dict(leverage_dist, orient='index', columns=['Value']))

# Leverage by GCF instrument
inst_lev = (gcf_fund.merge(df[['approvedref','cofinancing_leverage']], on='approvedref', how='left')
            .groupby('instrument')['cofinancing_leverage']
            .agg(Projects='count', Median='median', Mean='mean',
                 P25=lambda x: x.quantile(0.25),
                 P75=lambda x: x.quantile(0.75))
            .round(2))
display(HTML('<h4>Leverage Distribution by GCF Instrument</h4>'))
display(inst_lev)

# Leverage by region
lev_region = df.groupby('region')['cofinancing_leverage'].agg(
    Projects='count', Median='median', Mean='mean', Std='std'
).round(2).sort_values('Median', ascending=False)
display(HTML('<h4>Leverage by Region</h4>'))
display(lev_region)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ADD YOUR OWN RISK EQ — duplicate this cell
# ═══════════════════════════════════════════════════════════════════════
EQ = 'Risk EQ_: [Your question here]'

eq_df = df.copy()
# eq_df = eq_df[eq_df['status'] == 'Under Implementation']   # filter

indicators = {
    'N projects':          len(eq_df),
    'Total GCF (USD M)':   round(eq_df['totalgcffunding'].sum()/1e6, 1),
    'Avg GCF (USD M)':     round(eq_df['totalgcffunding'].mean()/1e6, 1),
    'Median leverage':     round(eq_df['cofinancing_leverage'].median(), 2),
    'HHI (country)':       hhi(eq_df.groupby('country_name')['totalgcffunding'].sum().values),
    'Gini':                round(gini(eq_df['totalgcffunding'].dropna().values), 4),
}

display(HTML(f'<h3>{EQ}</h3>'))
display(pd.DataFrame.from_dict(indicators, orient='index', columns=['Value']))
portfolio_table(eq_df, 'region', label=f'{EQ} — by Region')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ6: Contingent Liability Risk Matrix
#
# Framework: IMF four-quadrant contingent liability classification
# (Explicit/Implicit × Direct/Contingent)
# Source: "Analyzing and Managing Fiscal Risks — Best Practices" (IMF, 2016)
#   https://www.imf.org/external/np/pp/eng/2016/050416.pdf
# Also: "Contingent Government Liabilities: A Hidden Fiscal Risk" (IMF F&D, 1999)
#   https://www.imf.org/external/pubs/ft/fandd/1999/03/polackov.htm
#
# Key IMF insight: direct fiscal cost of contingent liability realisation
# averages 6% of GDP, but debt-to-GDP impact averages 15% due to correlated
# macro shocks triggering multiple liabilities simultaneously.
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ6: Contingent Liability Risk Matrix (IMF Framework)</h3>'))

# ── IMF four-quadrant classification of GCF instruments ───────────────
# Explicit-Direct:      Grants, disbursed Senior Loans — certain, legally binding
# Explicit-Contingent:  Guarantees, Subordinated Loans, Equity — depend on trigger events
# Implicit-Direct:      Reimbursable Grants — moral obligation to recover
# Implicit-Contingent:  Results-Based Payments — contingent on outcome achievement

quadrant_map = {
    'Grants':              ('Explicit',  'Direct',      'Low',    'Non-repayable; full cost recognised upfront'),
    'Senior Loans':        ('Explicit',  'Direct',      'Medium', 'Repayment obligation; credit risk via PD/LGD'),
    'Reimbursable Grants': ('Implicit',  'Direct',      'Medium', 'Moral recovery obligation; often written off'),
    'Subordinated Loans':  ('Explicit',  'Contingent',  'High',   'Junior claim; absorbs first loss in default'),
    'Equity':              ('Explicit',  'Contingent',  'High',   'Mark-to-market; illiquid in EM; first-loss'),
    'Guarantees':          ('Explicit',  'Contingent',  'High',   'Contingent call; correlated with macro shocks'),
    'Results-Based Payment':('Implicit', 'Contingent',  'Medium', 'Disbursed only upon verified outcome delivery'),
    'in-kind':             ('Implicit',  'Direct',      'Low',    'Non-financial; operational cost only'),
}

quad_df = pd.DataFrame.from_dict(quadrant_map, orient='index',
    columns=['Liability_Type', 'Timing', 'Tail_Risk', 'Notes'])
display(HTML('<h4>Instrument Classification by IMF Quadrant</h4>'))
display(quad_df)

# ── Map classification onto instrument funding table ───────────────────
if len(gcf_fund):
    gcf_fund['liability_type'] = gcf_fund['instrument'].map(
        {k: v[0] for k, v in quadrant_map.items()}).fillna('Explicit')
    gcf_fund['timing'] = gcf_fund['instrument'].map(
        {k: v[1] for k, v in quadrant_map.items()}).fillna('Direct')
    gcf_fund['tail_risk'] = gcf_fund['instrument'].map(
        {k: v[2] for k, v in quadrant_map.items()}).fillna('Medium')

    # Exposure by quadrant
    quad_exp = gcf_fund.groupby(['liability_type', 'timing']).agg(
        Instruments=('instrument', lambda x: ', '.join(x.unique())),
        Projects=('approvedref', 'nunique'),
        GCF_M=('budget_usd', lambda x: round(x.sum()/1e6, 1)),
        Share_pct=('budget_usd', lambda x: round(x.sum()/gcf_fund['budget_usd'].sum()*100, 1)),
    )
    display(HTML('<h4>GCF Exposure by IMF Quadrant (USD M)</h4>'))
    display(quad_exp)

    # Tail-risk exposure summary
    tail_exp = gcf_fund.groupby('tail_risk').agg(
        GCF_M=('budget_usd', lambda x: round(x.sum()/1e6, 1)),
        Share_pct=('budget_usd', lambda x: round(x.sum()/gcf_fund['budget_usd'].sum()*100, 1)),
        Projects=('approvedref', 'nunique'),
    ).reindex(['Low','Medium','High'])
    display(HTML('<h4>GCF Exposure by Tail-Risk Category</h4>'))
    display(tail_exp)

    # IMF fiscal multiplier: estimate tail debt impact
    contingent_m = gcf_fund[gcf_fund['timing']=='Contingent']['budget_usd'].sum()/1e6
    total_m      = gcf_fund['budget_usd'].sum()/1e6
    display(HTML(
        f'<p><b>IMF Fiscal Multiplier Alert:</b> Contingent exposure = '
        f'<b>USD {contingent_m:,.1f}M ({contingent_m/total_m*100:.1f}% of GCF deployed)</b>. '
        f'Per IMF best practices, correlated macro shocks can amplify direct fiscal cost '
        f'(avg 6% GDP) to 2.5x that level in debt impact. '
        f'<i>Source: IMF (2016) https://www.imf.org/external/np/pp/eng/2016/050416.pdf</i></p>'
    ))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ7: Sovereign Fiscal Stress Overlay
#
# Framework: IMF Debt Sustainability Analysis (DSA) risk ratings
# Source: IMF Debt Sustainability Framework for Low-Income Countries
#   https://www.imf.org/external/pubs/ft/dsa/lic.htm
# Source: IMF Sovereign Risk and Debt Sustainability Analysis (MAC SRDSA, 2021)
#   https://www.imf.org/en/Publications/DSA/sovereign-risk-and-debt-sustainability-analysis-for-market-access-countries
# Source: IMF Fiscal Risk Assessment Tool (FRAT)
#   https://www.imf.org/en/Topics/fiscal-policies/Fiscal-Risks/Fiscal-Risks-Toolkit/Fiscal-Risks-Toolkit-FRAT
#
# Methodology: Cross-reference GCF country exposures against IMF DSA
# risk ratings to produce a credit-quality-weighted exposure map.
# Four risk categories: Low / Moderate / High / In Distress
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ7: Sovereign Fiscal Stress Overlay (IMF DSA Framework)</h3>'))

# ── IMF DSA risk ratings (as of latest IMF/World Bank assessments) ────
# Source: IMF Debt Sustainability Analysis country pages
# https://www.imf.org/external/pubs/ft/dsa/
# Update this lookup periodically from: https://www.imf.org/en/Publications/DSA
# Ratings: 'Low', 'Moderate', 'High', 'In Distress'

dsa_ratings = {
    # Africa
    'Ethiopia':             'In Distress',
    'Zambia':               'In Distress',
    'Ghana':                'In Distress',
    'Mozambique':           'High',
    'Tanzania':             'Moderate',
    'Kenya':                'High',
    'Rwanda':               'Moderate',
    'Senegal':              'Moderate',
    'Uganda':               'Moderate',
    'Cameroon':             'High',
    'Madagascar':           'High',
    'Niger':                'High',
    'Mali':                 'High',
    'Burkina Faso':         'High',
    'Democratic Republic of the Congo': 'High',
    'Malawi':               'In Distress',
    # Asia Pacific
    'Bangladesh':           'Moderate',
    'Cambodia':             'Moderate',
    'Myanmar':              'High',
    'Nepal':                'Low',
    'Pakistan':             'High',
    'Papua New Guinea':     'High',
    'Solomon Islands':      'High',
    'Vanuatu':              'Moderate',
    'Fiji':                 'Moderate',
    'Indonesia':            'Low',
    'Philippines':          'Low',
    'Vietnam':              'Moderate',
    # Latin America & Caribbean
    'Haiti':                'High',
    'Honduras':             'Moderate',
    'Bolivia':              'Moderate',
    'Ecuador':              'High',
    'Peru':                 'Low',
    'Colombia':             'Low',
    'Mexico':               'Low',
    'Jamaica':              'Moderate',
    'Belize':               'High',
    # Eastern Europe / Central Asia
    'Tajikistan':           'High',
    'Kyrgyz Republic':      'High',
    'Georgia':              'Moderate',
    # MENA
    'Jordan':               'High',
    'Morocco':              'Moderate',
    'Egypt':                'High',
    'Tunisia':              'High',
    'Yemen':                'In Distress',
}

dsa_order   = ['Low', 'Moderate', 'High', 'In Distress']
dsa_colors  = {'Low': 'green', 'Moderate': 'goldenrod', 'High': 'orange', 'In Distress': 'red'}

# Map DSA ratings onto country exposures
country_exp = df.groupby('country_name')['totalgcffunding'].sum().reset_index()
country_exp.columns = ['country_name', 'gcf_m']
country_exp['gcf_m'] = country_exp['gcf_m'] / 1e6
country_exp['dsa_rating'] = country_exp['country_name'].map(dsa_ratings).fillna('Not rated / MAC')
country_exp = country_exp.sort_values('gcf_m', ascending=False)

display(HTML('<h4>GCF Country Exposure with IMF DSA Risk Rating (Top 30)</h4>'))
display(country_exp.head(30).reset_index(drop=True))

# ── Credit-quality-weighted exposure summary ───────────────────────────
dsa_summary = country_exp.groupby('dsa_rating').agg(
    Countries=('country_name', 'count'),
    GCF_M=('gcf_m', lambda x: round(x.sum(), 1)),
    Share_pct=('gcf_m', lambda x: round(x.sum()/country_exp['gcf_m'].sum()*100, 1)),
    Avg_M=('gcf_m', lambda x: round(x.mean(), 1)),
    Largest_M=('gcf_m', 'max'),
)
# Reorder by risk level
existing = [r for r in dsa_order + ['Not rated / MAC'] if r in dsa_summary.index]
dsa_summary = dsa_summary.reindex(existing)
display(HTML(
    '<h4>Credit-Quality-Weighted Exposure (IMF DSA Categories)</h4>'
    '<p><i>Framework: IMF Debt Sustainability Framework — '
    'https://www.imf.org/external/pubs/ft/dsa/lic.htm</i></p>'
))
display(dsa_summary)

# ── At-risk exposure: High + In Distress ──────────────────────────────
at_risk = country_exp[country_exp['dsa_rating'].isin(['High', 'In Distress'])]
display(HTML(
    f'<p><b>Credit-Stressed Exposure:</b> '
    f'USD {at_risk["gcf_m"].sum():,.1f}M across {len(at_risk)} countries rated '
    f'High Risk or In Distress ({at_risk["gcf_m"].sum()/country_exp["gcf_m"].sum()*100:.1f}% of rated portfolio). '
    f'<i>Source: IMF DSA https://www.imf.org/external/pubs/ft/dsa/</i></p>'
))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# RISK EQ8: Co-financing Correlation & Shock Amplification Risk
#
# Framework: IMF fiscal stress multiplier principle — macro shocks trigger
# correlated, simultaneous crystallisation of multiple contingent liabilities.
# Source: "Measuring Concentration Risk — A Partial Portfolio Approach" (IMF WP/16/158)
#   https://www.imf.org/external/pubs/ft/wp/2016/wp16158.pdf
# Source: "Name Concentration Risk in MDB Sovereign Portfolios" (2024)
#   https://arxiv.org/html/2311.13802
# Source: G20 MDB Capital Adequacy Framework (2016)
#   http://www.g20chn.org/English/Documents/Current/201608/P020160811554323564059.pdf
#
# Key MDB finding: concentration risk = 46-82% of total unexpected loss
# in MDB sovereign portfolios (vs 1-8% for commercial banks).
# Co-financing that appears uncorrelated may be highly correlated during
# climate shocks — a property unique to climate finance portfolios.
# ═══════════════════════════════════════════════════════════════════════
display(HTML('<h3>Risk EQ8: Co-financing Correlation & Shock Amplification</h3>'))

# ── Co-financing by source: GCF vs co-financiers per country ──────────
if len(cofin_df):
    cofin_country = cofin_df.groupby('country_name')['budget_usd'].sum().reset_index()
    cofin_country.columns = ['country_name', 'cofin_m']
    cofin_country['cofin_m'] /= 1e6

    gcf_country = gcf_fund.groupby('country_name')['budget_usd'].sum().reset_index()
    gcf_country.columns = ['country_name', 'gcf_m']
    gcf_country['gcf_m'] /= 1e6

    combined = gcf_country.merge(cofin_country, on='country_name', how='outer').fillna(0)
    combined['total_m']   = combined['gcf_m'] + combined['cofin_m']
    combined['leverage']  = combined['cofin_m'] / combined['gcf_m'].replace(0, np.nan)
    combined['dsa_rating'] = combined['country_name'].map(dsa_ratings).fillna('Not rated / MAC')
    combined = combined.sort_values('total_m', ascending=False)

    display(HTML('<h4>GCF vs Co-financing Exposure by Country (Top 20)</h4>'))
    display(combined.head(20).round(2).reset_index(drop=True))

    # ── Leverage by DSA risk rating — does higher stress = lower crowding-in? ─
    lev_dsa = combined.groupby('dsa_rating')['leverage'].agg(
        Countries='count',
        Median_leverage='median',
        Mean_leverage='mean',
        Min_leverage='min',
        Max_leverage='max',
    ).round(2)
    existing = [r for r in dsa_order + ['Not rated / MAC'] if r in lev_dsa.index]
    lev_dsa = lev_dsa.reindex(existing)
    display(HTML(
        '<h4>Co-financing Leverage by IMF DSA Risk Rating</h4>'
        '<p><i>IMF MDB finding: co-financing appears uncorrelated in normal conditions '
        'but converges during climate/macro shocks — a concentration risk underestimated '
        'by standard models. Source: IMF WP/16/158 '
        'https://www.imf.org/external/pubs/ft/wp/2016/wp16158.pdf</i></p>'
    ))
    display(lev_dsa)

    # ── Shock scenario: what if co-financing fails in High/In Distress countries? ─
    stressed = combined[combined['dsa_rating'].isin(['High', 'In Distress'])]
    cofin_at_risk_m  = stressed['cofin_m'].sum()
    gcf_residual_m   = stressed['gcf_m'].sum()
    total_at_risk_m  = stressed['total_m'].sum()

    display(HTML(
        f'<h4>Stress Scenario: Co-financing Withdrawal in High/Distress Countries</h4>'
        f'<p>If co-financiers withdraw from High/In Distress countries during a climate shock:</p>'
        f'<ul>'
        f'<li>Co-financing at risk: <b>USD {cofin_at_risk_m:,.1f}M</b></li>'
        f'<li>GCF residual exposure: <b>USD {gcf_residual_m:,.1f}M</b></li>'
        f'<li>Total programme value at risk: <b>USD {total_at_risk_m:,.1f}M</b></li>'
        f'<li>Implied leverage collapse: from {stressed["leverage"].median():.1f}x to 0x</li>'
        f'</ul>'
        f'<p><i>Methodology: IMF fiscal multiplier principle — '
        f'correlated shock crystallisation. '
        f'Source: G20 MDB Capital Adequacy Framework (2016) '
        f'http://www.g20chn.org/English/Documents/Current/201608/P020160811554323564059.pdf</i></p>'
    ))
else:
    display(HTML('<i>Co-financing table not available — run Risk Setup cell first.</i>'))


## Section 5 — FT-Style Visualizations

In [ ]:
# ── FT Theme ──────────────────────────────────────────────────────────────
FT = {
    'bg':     '#FFF1E5',
    'blue':   '#0F5499',
    'red':    '#990F3D',
    'orange': '#FF8833',
    'teal':   '#0D7680',
    'grey':   '#66605C',
    'grid':   '#E0D5C9',
    'font':   'Arial',
}
FT_PALETTE = [FT['blue'], FT['red'], FT['orange'], FT['teal'], '#593380', '#006F9B']

def ft_base(figsize=(10, 5.5), title='', subtitle='', source='GCF Projects API | api.gcfund.org'):
    fig, ax = plt.subplots(figsize=figsize, facecolor=FT['bg'])
    ax.set_facecolor(FT['bg'])
    for sp in ['top', 'right', 'left']: ax.spines[sp].set_visible(False)
    ax.spines['bottom'].set_color(FT['grey'])
    ax.yaxis.grid(True, color=FT['grid'], linewidth=0.6, zorder=0)
    ax.xaxis.grid(False)
    ax.tick_params(colors=FT['grey'], labelsize=9)
    if title:
        fig.text(0.02, 0.97, title, ha='left', va='top',
                 fontsize=13, fontweight='bold', color='#1A1A1A', fontfamily=FT['font'])
    if subtitle:
        fig.text(0.02, 0.91, subtitle, ha='left', va='top',
                 fontsize=9.5, color=FT['grey'], fontfamily=FT['font'])
    if source:
        fig.text(0.02, 0.01, f'Source: {source}', ha='left', va='bottom',
                 fontsize=7.5, color=FT['grey'], fontfamily=FT['font'])
    fig.subplots_adjust(top=0.82, bottom=0.12)
    return fig, ax

print('FT theme ready.')

In [ ]:
# ── Chart 1: Top 15 countries by GCF commitment ───────────────────────────
top_n = 15
top = (df.groupby('country_name')['totalgcffunding']
         .sum().nlargest(top_n).sort_values() / 1e6)

fig, ax = ft_base(figsize=(9, top_n * 0.45 + 1.5),
                  title=f'Top {top_n} Recipients by GCF Commitment',
                  subtitle='USD millions — all approved projects')
bars = ax.barh(top.index, top.values, color=FT['blue'], height=0.65, zorder=3)
for bar, val in zip(bars, top.values):
    ax.text(val + top.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}M', va='center', ha='left', fontsize=8,
            color=FT['grey'], fontfamily=FT['font'])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
ax.yaxis.grid(False)
ax.xaxis.grid(True, color=FT['grid'], linewidth=0.6, zorder=0)
plt.tight_layout(rect=[0, 0.04, 1, 0.88])
plt.savefig(OUT_DIR / 'chart_top_countries.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
plt.show()

In [ ]:
# ── Chart 2: Stacked bar — GCF approvals by year, split Mitigation/Adaptation ─
pivot = (df.groupby(['approval_year', 'theme'])['totalgcffunding']
           .sum().unstack(fill_value=0) / 1e6).dropna()
pivot = pivot[[c for c in pivot.columns if pivot[c].sum() > 0]]

fig, ax = ft_base(figsize=(11, 5.5),
                  title='GCF Approvals by Year — Mitigation vs Adaptation vs Cross-cutting',
                  subtitle='USD millions')
bottom = np.zeros(len(pivot))
x = np.arange(len(pivot))
for col, color in zip(pivot.columns, FT_PALETTE):
    ax.bar(x, pivot[col], bottom=bottom, label=col, color=color, width=0.6, zorder=3)
    bottom += pivot[col].values
ax.set_xticks(x)
ax.set_xticklabels(pivot.index.astype(int), fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:,.0f}M'))
ax.legend(frameon=False, fontsize=8.5, labelcolor=FT['grey'])
plt.tight_layout(rect=[0, 0.04, 1, 0.88])
plt.savefig(OUT_DIR / 'chart_approvals_by_year.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
plt.show()

In [ ]:
# ── Chart 3: Scatter — project size vs co-financing leverage (by theme) ───
sc = df.dropna(subset=['totalgcffunding', 'cofinancing_leverage']).copy()
sc = sc[sc['cofinancing_leverage'] < sc['cofinancing_leverage'].quantile(0.97)]  # trim outliers

fig, ax = ft_base(figsize=(10, 6),
                  title='GCF Commitment vs Co-financing Leverage',
                  subtitle='Each dot = one project; size proportional to total beneficiaries')

for theme, color in zip(sc['theme'].unique(), FT_PALETTE):
    sub = sc[sc['theme'] == theme]
    sizes = np.sqrt(sub['total_beneficiaries'].clip(lower=1000)) / 5
    ax.scatter(sub['totalgcffunding'] / 1e6, sub['cofinancing_leverage'],
               s=sizes, alpha=0.6, color=color, label=theme, zorder=3)

ax.set_xlabel('GCF Commitment (USD M)', color=FT['grey'], fontsize=9)
ax.set_ylabel('Co-financing leverage (x)', color=FT['grey'], fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
ax.legend(frameon=False, fontsize=8.5, labelcolor=FT['grey'])
plt.tight_layout(rect=[0, 0.04, 1, 0.88])
plt.savefig(OUT_DIR / 'chart_leverage_scatter.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
plt.show()

In [ ]:
# ── Chart 4: LDC / SIDS / Other breakdown by region ──────────────────────
if 'is_ldc' in df.columns and 'region' in df.columns:
    df['vulnerability'] = 'Other'
    df.loc[df['is_sids'] == True, 'vulnerability'] = 'SIDS'
    df.loc[df['is_ldc'] == True, 'vulnerability'] = 'LDC'

    piv = df.groupby(['region', 'vulnerability']).size().unstack(fill_value=0)
    piv = piv.loc[piv.sum(axis=1).sort_values(ascending=True).index]

    fig, ax = ft_base(figsize=(10, 5.5),
                      title='Projects by Region — LDC, SIDS, Other',
                      subtitle='Number of approved projects')
    left = np.zeros(len(piv))
    cats = [c for c in ['LDC', 'SIDS', 'Other'] if c in piv.columns]
    for cat, color in zip(cats, FT_PALETTE):
        ax.barh(piv.index, piv[cat], left=left, color=color, label=cat, height=0.6, zorder=3)
        left += piv[cat].values
    ax.yaxis.grid(False)
    ax.xaxis.grid(True, color=FT['grid'], linewidth=0.6, zorder=0)
    ax.legend(frameon=False, fontsize=8.5, labelcolor=FT['grey'])
    plt.tight_layout(rect=[0, 0.04, 1, 0.88])
    plt.savefig(OUT_DIR / 'chart_ldc_sids_region.png', dpi=220, bbox_inches='tight', facecolor=FT['bg'])
    plt.show()

## Section 6 — Narrative Generation + Export

In [ ]:
def build_narrative(df):
    n = len(df)
    gcf_total_m = df['totalgcffunding'].sum() / 1e6
    gcf_avg_m   = df['totalgcffunding'].mean() / 1e6
    gcf_med_m   = df['totalgcffunding'].median() / 1e6
    lev         = df['cofinancing_leverage'].median()
    n_countries = df['country_name'].nunique() if 'country_name' in df.columns else 'N/A'
    ldc_n       = int(df['is_ldc'].sum()) if 'is_ldc' in df.columns else 'N/A'
    sids_n      = int(df['is_sids'].sum()) if 'is_sids' in df.columns else 'N/A'
    region_vc   = df['region'].value_counts() if 'region' in df.columns else None
    top_region  = region_vc.idxmax() if (region_vc is not None and len(region_vc)) else 'N/A'
    top_r_n     = int(region_vc.max()) if (region_vc is not None and len(region_vc)) else 0
    ben_m       = df['total_beneficiaries'].sum() / 1e6
    co2_mt      = df['lifetimeco2'].sum() / 1e6
    psf_n       = len(df[df['sector'].str.lower() == 'private']) if 'sector' in df.columns else 'N/A'

    by_status = df['status'].value_counts()
    status_text = ' | '.join([f'{s}: {v} ({v/n*100:.0f}%)' for s, v in by_status.items()])

    md = f"""## GCF Portfolio — Key Findings
*Generated: {datetime.now():%Y-%m-%d %H:%M} | Source: api.gcfund.org/v1/projects*

### Portfolio Scale
As of the data extraction date, the GCF portfolio comprises **{n:,} approved projects** across
**{n_countries} countries and territories**, with total GCF commitment of
**USD {gcf_total_m:,.1f} million**. The mean project size is USD {gcf_avg_m:,.1f}M
(median: USD {gcf_med_m:,.1f}M), reflecting a portfolio spanning micro to large-scale interventions.

### Co-financing Leverage
The median co-financing leverage ratio is **{lev:.2f}x**, meaning every USD 1 of GCF funding
mobilises USD {lev:.2f} in co-financing, consistent with GCF's catalytic mandate.

### Geographic Reach and Vulnerability Focus
The largest regional allocation is **{top_region}** with {top_r_n} projects
({top_r_n/n*100:.1f}% of portfolio). **{ldc_n} projects** target Least Developed Countries (LDCs)
and **{sids_n} projects** target Small Island Developing States (SIDS),
reflecting the GCF's mandated prioritisation of the most vulnerable nations.

### Implementation Status
{status_text}.

### Private Sector Engagement
**{psf_n} projects** are channelled through the Private Sector Facility (PSF),
representing GCF's engagement with non-sovereign financing modalities.

### Expected Climate Impact
The portfolio is projected to reduce or avoid **{co2_mt:,.1f} MtCO\u2082e** over project lifetimes
and reach an estimated **{ben_m:,.1f} million beneficiaries** (direct + indirect).

---
*Auto-generated narrative scaffold. Expand with evaluation judgements, caveats, and context.*
"""
    return md

narrative = build_narrative(df)
display(Markdown(narrative))

report_path = OUT_DIR / f'gcf_narrative_{datetime.now():%Y%m%d}.md'
report_path.write_text(narrative, encoding='utf-8')
print(f'Saved: {report_path}')


In [ ]:
# ── Export to Excel ────────────────────────────────────────────────────────
export_cols = [c for c in df.columns
               if c not in ['countries', 'entities', 'disbursements', 'funding', 'resultareas']]
export_path = OUT_DIR / f'gcf_projects_{datetime.now():%Y%m%d}.xlsx'

# Strip timezone from datetime columns before Excel export
export_df = df[export_cols].copy()
for col in export_df.select_dtypes(include=['datetimetz']).columns:
    export_df[col] = export_df[col].dt.tz_localize(None)

with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    export_df.to_excel(writer, sheet_name='All Projects', index=False)
    df.groupby('region').agg(
        Projects=('projectsid','count'),
        GCF_M=('totalgcffunding', lambda x: round(x.sum()/1e6,1))
    ).to_excel(writer, sheet_name='By Region')
    df.groupby('theme').agg(
        Projects=('projectsid','count'),
        GCF_M=('totalgcffunding', lambda x: round(x.sum()/1e6,1))
    ).to_excel(writer, sheet_name='By Theme')
    df.groupby('sector').agg(
        Projects=('projectsid','count'),
        GCF_M=('totalgcffunding', lambda x: round(x.sum()/1e6,1))
    ).to_excel(writer, sheet_name='By Sector')
    df.groupby('status').agg(
        Projects=('projectsid','count')
    ).to_excel(writer, sheet_name='By Status')
    trend.to_excel(writer, sheet_name='Annual Trend')

print(f'Exported: {export_path}')